# GraphRAG, explained: local Ollama + optional LangSmith tracing

This notebook explains the architecture in `Originals/graph_rag.ipynb` and provides a runnable local-model version. It keeps the original design: split documents into chunks, embed them, build a weighted graph over similar chunks, use spaCy/NLTK plus an LLM to identify concepts, retrieve starting chunks, traverse graph neighbors, check/generate an answer, and visualize the path.

> **What the original graph represents:** nodes are text chunks, and edges connect chunks whose embeddings are similar (with shared concepts contributing to the edge weight). This is a chunk-similarity graph, not an explicit entity-relation graph. The separate `graph_rag_local_attribution.ipynb` demonstrates a different entity/relationship graph design.

## Pipeline map

| Part | Purpose | Local implementation |
|---|---|---|
| 1. Runtime and settings | Select Ollama chat/embedding models; optionally enable LangSmith | `ChatOllama`, `OllamaEmbeddings`, environment variables |
| 2. NLP setup | Word tokenization/lemmatization and named-entity recognition | NLTK `WordNetLemmatizer`/`word_tokenize`, spaCy `en_core_web_sm` |
| 3. Document processor | Split documents and build a FAISS vector index | `RecursiveCharacterTextSplitter`, local Ollama embeddings |
| 4. Knowledge graph | Create chunk nodes, extract concepts, link similar chunks | NetworkX, spaCy entities, Ollama concept extraction, cosine similarity |
| 5. Query engine | Seed from vector retrieval, expand through weighted neighbors, answer from context | FAISS relevance search, priority queue, Ollama |
| 6. Visualization and orchestration | Show the traversal and expose a simple `ask` operation | Matplotlib/NetworkX, `GraphRAG` |

Ollama handles chat generation and embeddings locally. LangSmith is a separate, optional hosted tracing service: **if enabled, traces can include questions, prompts, retrieved context, and model answers.** Keep it disabled for sensitive corpora, or review your tracing/data-retention settings before enabling it.

## 1. Install dependencies and prepare local models

Install the packages listed in [`requirements-graph-rag-local.txt`](requirements-graph-rag-local.txt) into the same Python environment selected by your notebook kernel. Ollama is already configured with `qwen2.5:3b` on this machine, so use that local chat model; there is no need to pull Llama 3.1. Verify installed models with `ollama list`.

```powershell
ollama list
ollama pull nomic-embed-text
```

`nomic-embed-text` is an embedding model, separate from the Qwen chat model. The current local model list contains Qwen but no embedding model, so pull the embedding model above once (or set `OLLAMA_EMBEDDING_MODEL` to an embedding model you already have). A chat model cannot be used as an embedding model just by changing its name. For spaCy's English pipeline, install the model once in the notebook's Python environment:

```powershell
python -m spacy download en_core_web_sm
```

### Choosing Qwen2.5:3B or Llama 3.1

| Model | Advantages | Trade-offs | Good fit |
|---|---|---|---|
| **Qwen2.5:3B** (already installed) | Small footprint, faster responses, lower memory use; good for local prototypes, simple concept extraction, and routine questions. The Qwen family also has broad multilingual support. | A 3B model has less capacity for subtle multi-hop reasoning, consistent entity extraction, and strict structured output than a larger model. Validate JSON extraction and answer quality on your own documents. | Use first for this application when local resource use, privacy, and quick iteration matter. |
| **Llama 3.1 8B or larger** | Larger variants can provide stronger general instruction following, answer synthesis, and difficult multi-step reasoning. | More memory and compute, slower responses; exact quality and context behavior depend on the variant and quantization. It is not installed here and would require a separate download. | Consider only if Qwen's extraction or multi-hop answers are inadequate and your hardware can run the larger variant. |

For this GraphRAG pipeline, start with the installed Qwen model. Concept extraction and answer-completeness checks are repeated across chunks, so the smaller model can be substantially more responsive. If answers are weak, compare the models on the same small evaluation set (extracted concepts, JSON validity, retrieval path, and answer correctness) before downloading a larger model.

No OpenAI key is needed. LangSmith is optional and is only enabled when both `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true` are set in the process environment or a local `.env` file. Never commit `.env` or paste real keys into a notebook.

In [ ]:
from __future__ import annotations

import heapq
import os
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import networkx as nx
import nltk
import numpy as np
import spacy
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import traceable
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from pydantic import BaseModel, Field
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "qwen2.5:3b").strip() or "qwen2.5:3b"
EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
TRACE_REQUESTED = os.getenv("LANGSMITH_TRACING", "false").strip().lower() in {"1", "true", "yes"}
TRACING_ENABLED = TRACE_REQUESTED and bool(LANGSMITH_API_KEY)

if TRACE_REQUESTED and not LANGSMITH_API_KEY:
    raise RuntimeError("LangSmith tracing was requested but LANGSMITH_API_KEY is not set.")

os.environ["LANGSMITH_TRACING"] = "true" if TRACING_ENABLED else "false"
if TRACING_ENABLED:
    os.environ.setdefault("LANGSMITH_PROJECT", "local-graph-rag")

print(f"Chat model: {CHAT_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LangSmith tracing enabled: {TRACING_ENABLED}")

## 2. Initialize the local language models and NLP tools

`ChatOllama` is used for concept extraction, context sufficiency checks, and final answers. `OllamaEmbeddings` powers both FAISS retrieval and graph edge similarity; using the same embedding model across indexing and querying is important. NLTK tokenizes and lemmatizes extracted concepts, while spaCy contributes named entities from each chunk. The NLTK data downloads occur only if the resources are not already installed.

In [ ]:
for resource in ("punkt_tab", "wordnet"):
    if not nltk.download(resource, quiet=True):
        raise RuntimeError(f"Could not download required NLTK resource: {resource}")

try:
    nlp = spacy.load("en_core_web_sm")
except OSError as exc:
    raise RuntimeError(
        "spaCy model 'en_core_web_sm' is missing from this notebook kernel. "
        "Activate D:\\CUDA_ENV\\CUDA_ENV and run: python -m spacy download en_core_web_sm"
    ) from exc
lemmatizer = WordNetLemmatizer()

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)
embedding_model = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

print("Local LLM, embedding client, NLTK, and spaCy are ready.")

## 3. Load documents and split them into chunks

This notebook loads every PDF in `Notebooks/Documents` and preserves source/page metadata before splitting pages into overlapping chunks. Set `MAX_PAGES_PER_PDF = 10` for a smaller trial, or leave it as `None` to load every page. FAISS embeddings and graph embeddings are sent to Ollama in bounded batches to avoid oversized requests to the local runner.

Chunk overlap reduces the chance that a fact is lost at a boundary. Chunk size and overlap should be tuned for the model's context window and the structure of your documents.

In [ ]:
def load_pdf_documents(
    pdf_path: str | Path,
    max_pages: int | None = None,
) -> list[Document]:
    path = Path(pdf_path)
    if not path.is_file():
        raise FileNotFoundError(f"PDF does not exist: {path.resolve()}")
    loaded = PyPDFLoader(str(path)).load()
    return loaded if max_pages is None else loaded[:max_pages]


def load_pdf_folder(
    folder_path: str | Path,
    max_pages_per_pdf: int | None = None,
) -> list[Document]:
    folder = Path(folder_path)
    if not folder.is_dir():
        raise NotADirectoryError(f"Documents folder does not exist: {folder.resolve()}")

    pdf_paths = sorted(
        (p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"),
        key=lambda p: p.name.lower(),
    )
    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in: {folder.resolve()}")

    documents = []
    for pdf_path in pdf_paths:
        documents.extend(load_pdf_documents(pdf_path, max_pages_per_pdf))

    if not documents:
        raise ValueError(f"The PDFs in {folder.resolve()} contained no pages.")
    return documents


DOCUMENTS_DIR = Path(r"D:\CUDA_ENV\Graph_RAG_\Notebooks\Documents")
MAX_PAGES_PER_PDF = None  # None loads all pages; use 10 for a smaller trial.

documents = load_pdf_folder(DOCUMENTS_DIR, MAX_PAGES_PER_PDF)
print(f"Loaded {len(documents)} pages from {len({d.metadata['source'] for d in documents})} PDFs.")

In [ ]:
if not documents:
    raise ValueError("Provide at least one non-empty document.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
if not splits:
    raise ValueError("Document splitting produced no chunks.")

EMBEDDING_BATCH_SIZE = 16


def embed_documents_in_batches(texts: list[str], embedder, batch_size: int = EMBEDDING_BATCH_SIZE) -> list[list[float]]:
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")
    return [
        vector
        for start in range(0, len(texts), batch_size)
        for vector in embedder.embed_documents(texts[start : start + batch_size])
    ]


def build_faiss_index(documents: list[Document], embedder, batch_size: int = EMBEDDING_BATCH_SIZE):
    if not documents:
        raise ValueError("Cannot build a FAISS index without documents.")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    vector_store = FAISS.from_documents(documents[:batch_size], embedder)
    for start in range(batch_size, len(documents), batch_size):
        vector_store.add_documents(documents[start : start + batch_size])
    return vector_store


vector_store = build_faiss_index(splits, embedding_model)
print(f"Loaded {len(documents)} documents into {len(splits)} chunks and indexed them in FAISS.")

## 4. Build the chunk knowledge graph

Each split becomes a NetworkX node with its text, original metadata, and extracted concepts. spaCy supplies selected named entities; the Ollama model extracts general concepts as JSON. NLTK lemmatization normalizes concepts for later comparisons.

For every pair of chunks, cosine similarity is calculated over the local embeddings. An edge is added when similarity passes the threshold. Its weight combines semantic similarity and the fraction of shared concepts, using the original notebook's `0.7 / 0.3` weighting. This all-pairs comparison is quadratic in the number of chunks; for large corpora, use approximate-neighbor retrieval or a graph database rather than constructing a dense similarity matrix.

In [ ]:
class Concepts(BaseModel):
    concepts_list: list[str] = Field(description="Short, important general concepts")


class KnowledgeGraph:
    def __init__(self, language_model, nlp_pipeline, word_lemmatizer, edge_threshold: float = 0.8):
        self.graph = nx.Graph()
        self.llm = language_model
        self.nlp = nlp_pipeline
        self.lemmatizer = word_lemmatizer
        self.edges_threshold = edge_threshold
        self.concept_cache: dict[str, list[str]] = {}
        self.concept_parser = JsonOutputParser(pydantic_object=Concepts)
        self.concept_chain = (
            PromptTemplate(
                input_variables=["text"],
                partial_variables={"format_instructions": self.concept_parser.get_format_instructions()},
                template=(
                    "Extract concise key concepts, excluding named entities, from the text. "
                    "Return only the requested JSON.\n{format_instructions}\n\nText:\n{text}"
                ),
            )
            | self.llm.bind(format="json")
            | self.concept_parser
        )

    def _lemmatize_concept(self, concept: str) -> str:
        tokens = word_tokenize(concept.lower())
        return " ".join(self.lemmatizer.lemmatize(token) for token in tokens if token.isalpha())

    def _extract_concepts_and_entities(self, content: str) -> list[str]:
        if content in self.concept_cache:
            return self.concept_cache[content]

        named_entities = [
            entity.text
            for entity in self.nlp(content).ents
            if entity.label_ in {"PERSON", "ORG", "GPE", "WORK_OF_ART"}
        ]
        extracted = Concepts.model_validate(self.concept_chain.invoke({"text": content}))
        concepts = sorted({item.strip() for item in named_entities + extracted.concepts_list if item.strip()})
        self.concept_cache[content] = concepts
        return concepts

    def build(self, chunk_documents: list[Document], embedder) -> None:
        if not chunk_documents:
            raise ValueError("Cannot build a knowledge graph without chunks.")

        for node_id, chunk in enumerate(chunk_documents):
            self.graph.add_node(
                node_id,
                content=chunk.page_content,
                metadata=chunk.metadata,
                concepts=self._extract_concepts_and_entities(chunk.page_content),
            )

        texts = [item.page_content for item in chunk_documents]
        vectors = np.asarray(embed_documents_in_batches(texts, embedder))
        similarity_matrix = cosine_similarity(vectors)
        for left in range(len(chunk_documents)):
            left_concepts = set(self.graph.nodes[left]["concepts"])
            for right in range(left + 1, len(chunk_documents)):
                similarity = float(similarity_matrix[left, right])
                if similarity < self.edges_threshold:
                    continue

                right_concepts = set(self.graph.nodes[right]["concepts"])
                shared = left_concepts & right_concepts
                max_shared = min(len(left_concepts), len(right_concepts))
                shared_ratio = len(shared) / max_shared if max_shared else 0.0
                weight = 0.7 * similarity + 0.3 * shared_ratio
                self.graph.add_edge(
                    left,
                    right,
                    weight=weight,
                    similarity=similarity,
                    shared_concepts=sorted(shared),
                )

knowledge_graph = KnowledgeGraph(llm, nlp, lemmatizer)
knowledge_graph.build(splits, embedding_model)
print(f"Graph has {knowledge_graph.graph.number_of_nodes()} chunk nodes and {knowledge_graph.graph.number_of_edges()} edges.")

## 5. Retrieve, traverse, check, and answer

The query starts with FAISS's **relevance scores** (higher means more relevant), then maps those chunks to graph nodes. The original implementation inverted FAISS distance scores as if they were similarities; this version uses relevance scores and converts them to priority costs explicitly.

The priority queue starts from the most relevant chunks and expands over graph edges, where a stronger edge has a smaller traversal cost. After adding each chunk, the local model checks whether the accumulated context is sufficient. If it is not, the engine generates a final answer from the collected context. The context budget bounds prompt size. Unlike the cloud-oriented source notebook, this local walkthrough omits LLM compression of retrieved chunks; graph traversal and a bounded context provide the retrieval context without an additional model call per retrieved document.

In [ ]:
class AnswerCheck(BaseModel):
    is_complete: bool = Field(description="Whether the supplied context fully answers the question")
    answer: str = Field(description="A concise answer when the context is sufficient; otherwise an empty string")


class QueryEngine:
    def __init__(self, store, graph: KnowledgeGraph, language_model, max_context_chars: int = 4000):
        self.vector_store = store
        self.knowledge_graph = graph
        self.llm = language_model
        self.max_context_chars = max_context_chars
        self.answer_parser = JsonOutputParser(pydantic_object=AnswerCheck)
        self.answer_check_chain = (
            PromptTemplate(
                input_variables=["query", "context"],
                partial_variables={"format_instructions": self.answer_parser.get_format_instructions()},
                template=(
                    "Decide whether the context contains enough evidence to answer the question. "
                    "Do not use outside knowledge. If insufficient, set is_complete to false and answer to an empty string. "
                    "Return only JSON matching this schema.\n{format_instructions}\n\n"
                    "Question: {query}\n\nContext:\n{context}"
                ),
            )
            | self.llm.bind(format="json")
            | self.answer_parser
        )
        self.answer_chain = (
            PromptTemplate(
                input_variables=["query", "context"],
                template=(
                    "Answer the question using only the supplied context. If it does not contain "
                    "the answer, say that you do not know.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
                ),
            )
            | self.llm
            | StrOutputParser()
        )

    def _check_answer(self, query: str, context: str) -> AnswerCheck:
        return AnswerCheck.model_validate(self.answer_check_chain.invoke({"query": query, "context": context}))

    @traceable(name="graph_rag_query", run_type="chain")
    def query(self, query: str) -> dict[str, Any]:
        if not query.strip():
            raise ValueError("Query must not be empty.")
        graph = self.knowledge_graph.graph
        if graph.number_of_nodes() == 0:
            raise RuntimeError("The knowledge graph is empty; ingest documents before querying.")

        node_by_content = {data["content"]: node for node, data in graph.nodes(data=True)}
        retrieved = self.vector_store.similarity_search_with_relevance_scores(query, k=5)
        if not retrieved:
            raise RuntimeError("FAISS returned no relevant chunks for the query.")

        queue: list[tuple[float, int]] = []
        distances: dict[int, float] = {}
        for document, relevance in retrieved:
            node = node_by_content.get(document.page_content)
            if node is None:
                continue
            priority = 1.0 - min(1.0, max(0.0, float(relevance)))
            if priority < distances.get(node, float("inf")):
                distances[node] = priority
                heapq.heappush(queue, (priority, node))
        if not queue:
            raise RuntimeError("Retrieved chunks could not be matched to graph nodes.")

        traversal_path: list[int] = []
        context_parts: list[str] = []
        context_chars = 0
        final_answer = ""
        visited: set[int] = set()

        while queue and context_chars < self.max_context_chars:
            priority, node = heapq.heappop(queue)
            if node in visited or priority > distances.get(node, float("inf")):
                continue
            visited.add(node)
            content = graph.nodes[node]["content"]
            remaining = self.max_context_chars - context_chars
            content = content[:remaining]
            if not content:
                break

            traversal_path.append(node)
            context_parts.append(content)
            context_chars += len(content)
            context = "\n\n".join(context_parts)
            check = self._check_answer(query, context)
            if check.is_complete and check.answer.strip():
                final_answer = check.answer.strip()
                break

            for neighbor in graph.neighbors(node):
                if neighbor in visited:
                    continue
                edge_weight = max(float(graph.edges[node, neighbor]["weight"]), 1e-6)
                new_priority = priority + 1.0 / edge_weight
                if new_priority < distances.get(neighbor, float("inf")):
                    distances[neighbor] = new_priority
                    heapq.heappush(queue, (new_priority, neighbor))

        context = "\n\n".join(context_parts)
        if not final_answer:
            final_answer = self.answer_chain.invoke({"query": query, "context": context})

        return {
            "question": query,
            "answer": final_answer,
            "context": context,
            "traversal_path": traversal_path,
            "source_metadata": [graph.nodes[node]["metadata"] for node in traversal_path],
        }

query_engine = QueryEngine(vector_store, knowledge_graph, llm)
print("Query engine is ready.")

## 6. Visualize the graph and highlight the retrieval path

The visualization shows all chunk nodes and similarity edges; red arrows mark the order in which the query engine visited nodes. A graph may contain no edges if the corpus is small or the threshold is too high. In that case the vector-retrieved starting chunks can still provide context, but graph expansion cannot connect them. Inspect edge counts and tune the threshold only after checking retrieval quality.

In [ ]:
def visualize_traversal(graph: nx.Graph, traversal_path: list[int]) -> None:
    if graph.number_of_nodes() == 0:
        raise ValueError("Cannot visualize an empty graph.")

    positions = nx.spring_layout(graph, seed=7)
    fig, axis = plt.subplots(figsize=(11, 7))
    nx.draw_networkx_edges(graph, positions, edge_color="#9aa4b2", width=1.5, ax=axis)
    nx.draw_networkx_nodes(graph, positions, node_color="#b9d8f2", node_size=1000, ax=axis)
    labels = {
        node: f"{node}: {data['concepts'][0] if data['concepts'] else 'chunk'}"
        for node, data in graph.nodes(data=True)
    }
    nx.draw_networkx_labels(graph, positions, labels, font_size=8, ax=axis)

    if traversal_path:
        path_edges = list(zip(traversal_path, traversal_path[1:]))
        nx.draw_networkx_nodes(
            graph,
            positions,
            nodelist=traversal_path,
            node_color=["#79c99e" if node == traversal_path[0] else "#f2a0a0" if node == traversal_path[-1] else "#ffd166" for node in traversal_path],
            node_size=1200,
            ax=axis,
        )
        nx.draw_networkx_edges(
            graph,
            positions,
            edgelist=path_edges,
            edge_color="#d62728",
            width=2.5,
            arrows=True,
            ax=axis,
        )

    axis.set_title("Chunk-similarity graph and query traversal")
    axis.axis("off")
    plt.tight_layout()
    plt.show()

## 7. Ask a question and inspect its trace

The `query` method is decorated with LangSmith's `traceable`, and LangChain model/prompt calls made inside it are also traced when tracing is enabled. The returned object includes the exact question, answer, accumulated context, traversal node IDs, and source metadata. Use the `source_metadata` to connect retrieved chunks back to source documents/pages.

To enable traces, set `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true` outside this notebook or in an untracked local `.env` file, then restart the kernel. Tracing is disabled by default. If enabled, do not assume that Ollama's local inference means end-to-end privacy: LangSmith receives trace data over the network.

In [ ]:
question = "How do human activities contribute to rising global temperatures?"
result = query_engine.query(question)

print("Question:", result["question"])
print("\nAnswer:\n", result["answer"])
print("\nTraversal nodes:", result["traversal_path"])
print("\nSources:")
for source in result["source_metadata"]:
    print(source)

visualize_traversal(knowledge_graph.graph, result["traversal_path"])

## 8. Using the original PDF workflow

The document-loading cell reads all PDFs in `Notebooks/Documents` by default. To use a single local PDF instead, load it with `load_pdf_documents`; then build the FAISS index with the batched helper so no large embedding request is sent to Ollama:

```python
documents = load_pdf_documents(Path("Notebooks") / "Documents" / "your_document.pdf", max_pages=10)
splits = text_splitter.split_documents(documents)
vector_store = build_faiss_index(splits, embedding_model)
knowledge_graph = KnowledgeGraph(llm, nlp, lemmatizer, edge_threshold=0.8)
knowledge_graph.build(splits, embedding_model)
query_engine = QueryEngine(vector_store, knowledge_graph, llm)
```

Run those statements instead of the sample-corpus setup and rebuild the graph after changing documents. The PDF content, embeddings, concept extraction, graph construction, and answer generation stay on the local Ollama/Python stack; LangSmith tracing remains a separate opt-in data flow.

## Design notes and troubleshooting

- **Ollama connection errors:** confirm the Ollama service is running, the base URL is reachable, and both configured model names have been pulled.
- **JSON parsing errors:** use a chat model that honors Ollama JSON mode; the notebook intentionally raises parsing/model errors rather than silently substituting empty concepts or answers.
- **spaCy model errors:** run `python -m spacy download en_core_web_sm` in the notebook's Python environment, then restart the kernel.
- **Sparse graph:** the edge threshold is `0.8`, matching the original notebook. Smaller corpora or different embedding models may need a measured adjustment.
- **Large corpora:** the dense cosine-similarity matrix is quadratic. Use a nearest-neighbor index to generate candidate edges, and persist the FAISS index/graph before scaling up.
- **LangSmith traces missing:** confirm both tracing variables are available to the kernel and restart it. The notebook deliberately raises an error if tracing is requested without an API key.
- **Sensitive data:** keep tracing disabled unless sending prompts, retrieved text, and answers to LangSmith is acceptable for your data.

The original notebooks in `Originals/` are left unchanged.

## Windows virtual environment

Use the existing project environment for installation and notebook execution. From the project root, activate it in PowerShell, then install the notebook requirements into that activated interpreter:

```powershell
& "D:\CUDA_ENV\CUDA_ENV\Scripts\Activate.ps1"
python -m pip install -r Notebooks\requirements-graph-rag-local.txt
```

In VS Code, select `D:\CUDA_ENV\CUDA_ENV\Scripts\python.exe` as the notebook kernel before running cells. Activate the environment in the same terminal session where you run the notebook server.